Création de la base de données

In [ ]:
CREATE DATABASE IF NOT EXISTS LINKEDIN;
USE DATABASE LINKEDIN;

Création de la base de données

In [ ]:
USE DATABASE LINKEDIN;

CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD;

## Création du stage externe (Bucket S3 Public)
Note : Remplacez 's3://lienacopier/' par l'URL réelle fournie

In [ ]:
USE SCHEMA BRONZE;
CREATE OR REPLACE STAGE linkedin_stage
  URL = 's3://snowflake-lab-bucket/';

In [ ]:
LIST @linkedin.bronze.linkedin_stage

In [ ]:
USE DATABASE LINKEDIN;

USE SCHEMA BRONZE;

Fichier .CSV

In [ ]:
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
  type = 'CSV'
  field_delimiter = ','
  record_delimiter = '\n'
  skip_header = 1
  field_optionally_enclosed_by = '\042'
  null_if = (''); 

Fichier.json

In [ ]:
CREATE OR REPLACE FILE FORMAT json_format
TYPE = JSON;

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.BRONZE.job_postings (
    job_id STRING,
    company_name STRING,
    title STRING,
    description STRING,
    max_salary FLOAT,
    med_salary FLOAT,
    min_salary FLOAT,
    pay_period STRING,
    formatted_work_type STRING,
    location STRING,
    applies INT,
    original_listed_time STRING,
    remote_allowed BOOLEAN,
    views INT,
    job_posting_url STRING,
    application_url STRING,
    application_type STRING,
    expiry STRING,
    closed_time STRING,
    formatted_experience_level STRING,
    skills_desc STRING,
    listed_time STRING,
    posting_domain STRING,
    sponsored BOOLEAN,
    work_type STRING,
    currency STRING,
    compensation_type STRING
);

In [ ]:
CREATE OR REPLACE TABLE benefits (
    job_id STRING,
    inferred BOOLEAN,
    type STRING
);

In [ ]:
CREATE OR REPLACE TABLE employee_counts (
    company_id STRING,
    employee_count INT,
    follower_count INT,
    time_recorded STRING
);

In [ ]:
CREATE OR REPLACE TABLE job_skills (
    job_id STRING,
    skill_abr STRING
);

## Tables JSON (VARIANT)

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.BRONZE.companies (data VARIANT);
CREATE OR REPLACE TABLE LINKEDIN.BRONZE.company_industries (data VARIANT);
CREATE OR REPLACE TABLE LINKEDIN.BRONZE.company_specialities (data VARIANT);
CREATE OR REPLACE TABLE LINKEDIN.BRONZE.job_industries (data VARIANT);

# Chargement des données

## CSV

In [ ]:
COPY INTO LINKEDIN.BRONZE.job_postings
FROM (
    SELECT
        $1::STRING,  -- job_id
        $2::STRING,  -- company_name ✅ forcé en texte
        $3::STRING,
        $4::STRING,
        $5::FLOAT,
        $6::FLOAT,
        $7::FLOAT,
        $8::STRING,
        $9::STRING,
        $10::STRING,
        $11::INT,
        $12::STRING,
        $13::BOOLEAN,
        $14::INT,
        $15::STRING,
        $16::STRING,
        $17::STRING,
        $18::STRING,
        $19::STRING,
        $20::STRING,
        $21::STRING,
        $22::STRING,
        $23::STRING,
        $24::BOOLEAN,
        $25::STRING,
        $26::STRING,
        $27::STRING
    FROM @linkedin_stage/job_postings.csv
)
FILE_FORMAT = csv_format
ON_ERROR = CONTINUE;

COPY INTO LINKEDIN.BRONZE.benefits
FROM @linkedin_stage/benefits.csv
FILE_FORMAT = csv_format;

COPY INTO LINKEDIN.BRONZE.employee_counts
FROM @linkedin_stage/employee_counts.csv
FILE_FORMAT = csv_format
ON_ERROR = CONTINUE;

COPY INTO LINKEDIN.BRONZE.job_skills
FROM @linkedin_stage/job_skills.csv
FILE_FORMAT = csv_format;

##### Après vérification de la table, LINKEDIN.BRONZE.job_postings la colonne company_name est sous la forme numérique, peut-être la copie a été mal faite ou soit cette colonne comporte les company-id.

In [ ]:
UPDATE LINKEDIN.BRONZE.job_postings
SET company_name = TO_VARCHAR(TO_NUMBER(company_name));

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.SILVER.job_postings AS 
SELECT 
    job_id, 
    company_name AS company_id, 
    title, 
    description, 
    max_salary, 
    med_salary, 
    min_salary, 
    pay_period, 
    formatted_work_type, 
    location, 
    applies, 
    TO_TIMESTAMP(TO_NUMBER(original_listed_time)/1000) AS original_listed_ts,
    TO_TIMESTAMP(TO_NUMBER(listed_time)/1000) AS listed_time_ts, 
    TO_TIMESTAMP(TO_NUMBER(expiry)/1000) AS expiry_ts, 
    TO_TIMESTAMP(TO_NUMBER(closed_time)/1000) AS closed_time_ts, 
    remote_allowed, 
    views, 
    job_posting_url, 
    application_url, 
    application_type, 
    formatted_experience_level, 
    skills_desc, 
    posting_domain, 
    sponsored, 
    work_type, 
    currency, 
    compensation_type 
    
FROM LINKEDIN.BRONZE.job_postings;

In [ ]:
select * from LINKEDIN.BRONZE.job_postings;

In [ ]:
select * from LINKEDIN.BRONZE.benefits;

In [ ]:
CREATE OR REPLACE TABLE employee_counts AS
SELECT
    company_id,
    employee_count,
    follower_count,
    
    TO_TIMESTAMP(
        CAST(FLOOR(TO_DOUBLE(time_recorded)) AS INTEGER)
    ) AS time_recorded

FROM employee_counts;

In [ ]:
select * from LINKEDIN.BRONZE.employee_counts;

In [ ]:
select * from LINKEDIN.BRONZE.job_skills;

## JSON

In [ ]:
COPY INTO LINKEDIN.BRONZE.companies
FROM @linkedin_stage/companies.json
FILE_FORMAT = json_format;

COPY INTO LINKEDIN.BRONZE.company_industries
FROM @linkedin_stage/company_industries.json
FILE_FORMAT = json_format;

COPY INTO LINKEDIN.BRONZE.company_specialities
FROM @linkedin_stage/company_specialities.json
FILE_FORMAT = json_format;

COPY INTO LINKEDIN.BRONZE.job_industries
FROM @linkedin_stage/job_industries.json
FILE_FORMAT = json_format;

In [ ]:
select * from LINKEDIN.BRONZE.companies;

In [ ]:
select * from LINKEDIN.BRONZE.company_industries;

In [ ]:
select * from LINKEDIN.BRONZE.job_industries;

# Transformation des JSON en tables propres

##### Vérification du type de .JSON car, les tables générées étaient vides.

In [ ]:
SELECT data
FROM LINKEDIN.BRONZE.companies
LIMIT 5;

##### Le type de .json est "Array". Le code de reécriture de la table change alors. Au lieu d'utiliser "creat table select data : " ,il faut utiliser plutôt "creat table select value : " + Lateral flatten, comme ci-dessous.

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.SILVER.companies AS
SELECT
    value:company_id::STRING AS company_id,
    value:name::STRING AS name,
    value:description::STRING AS description,
    value:company_size::INT AS company_size,
    value:state::STRING AS state,
    value:country::STRING AS country,
    value:city::STRING AS city,
    value:zip_code::STRING AS zip_code,
    value:address::STRING AS address,
    value:url::STRING AS url

FROM LINKEDIN.BRONZE.companies,
LATERAL FLATTEN(input => data);

In [ ]:
Select * from LINKEDIN.SILVER.companies;

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.SILVER.company_industries AS
SELECT
    value:company_id::STRING AS company_id,
    value:industry::STRING AS industry
FROM LINKEDIN.BRONZE.company_industries,
LATERAL FLATTEN(input => data);;

In [ ]:
Select * from LINKEDIN.SILVER.company_industries;

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.SILVER.company_specialities AS
SELECT
    value:company_id::STRING AS company_id,
    value:speciality::STRING AS speciality
FROM LINKEDIN.BRONZE.company_specialities,
LATERAL FLATTEN(input => data);

In [ ]:
Select * from LINKEDIN.SILVER.company_specialities;

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.SILVER.job_industries AS
SELECT
    value:job_id::STRING AS job_id,
    value:industry_id::STRING AS industry_id
FROM LINKEDIN.BRONZE.job_industries,
LATERAL FLATTEN(input => data);;

In [ ]:
Select * from LINKEDIN.SILVER.job_industries;

## Nettoyage salaires

In [ ]:
UPDATE LINKEDIN.SILVER.job_postings
SET max_salary = NULL
WHERE max_salary = 0;

## Uniformisation remote

In [ ]:
UPDATE LINKEDIN.SILVER.job_postings
SET remote_allowed = FALSE
WHERE remote_allowed IS NULL;

In [ ]:
Select * from LINKEDIN.SILVER.job_postings;

## Déduplication

In [ ]:
CREATE OR REPLACE TABLE LINKEDIN.GOLD.job_postings AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY listed_time_ts DESC) AS rn
    FROM LINKEDIN.SILVER.job_postings
)
WHERE rn = 1;

In [ ]:
select * from LINKEDIN.GOLD.job_postings;

# Analyse des Données

## Top 10 des titres de postes par industrie

In [ ]:
SELECT *
FROM (
    SELECT 
        cs.speciality,
        j.title,
        COUNT(*) AS nb_jobs,
        ROW_NUMBER() OVER (
            PARTITION BY cs.speciality 
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM LINKEDIN.GOLD.job_postings j
    JOIN LINKEDIN.SILVER.company_specialities cs 
        ON j.company_id = cs.company_id
    GROUP BY cs.speciality, j.title
)
WHERE rank <= 10
ORDER BY speciality, rank;

In [ ]:
SELECT *
FROM (
    SELECT 
        cs.speciality,
        j.title,
        COUNT(*) AS nb_jobs,
        ROW_NUMBER() OVER (
            PARTITION BY cs.speciality 
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM LINKEDIN.GOLD.job_postings j
    JOIN LINKEDIN.SILVER.company_specialities cs 
        ON j.company_id = cs.company_id
    GROUP BY cs.speciality, j.title
) t
WHERE rank <= 10
  AND speciality IN (
      SELECT speciality
      FROM LINKEDIN.GOLD.job_postings j
      JOIN LINKEDIN.SILVER.company_specialities cs 
          ON j.company_id = cs.company_id
      GROUP BY speciality
      HAVING COUNT(DISTINCT title) > 10
  )
ORDER BY speciality, rank;

## Top 10 des jobs les mieux payés par industrie

In [ ]:
SELECT 
    cs.speciality,
    jp.title,
    MAX(jp.max_salary) AS max_salary
FROM LINKEDIN.GOLD.job_postings jp
JOIN LINKEDIN.SILVER.company_specialities cs 
    ON jp.company_id = cs.company_id
WHERE jp.max_salary IS NOT NULL
GROUP BY 1,2
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY cs.speciality 
    ORDER BY MAX(jp.max_salary) DESC
) <= 10;

In [ ]:
SELECT *
FROM (
    SELECT 
        cs.speciality,
        jp.title,
        MAX(jp.max_salary) AS max_salary,
        ROW_NUMBER() OVER (
            PARTITION BY cs.speciality 
            ORDER BY MAX(jp.max_salary) DESC
        ) AS rank
    FROM LINKEDIN.GOLD.job_postings jp
    JOIN LINKEDIN.SILVER.company_specialities cs 
        ON jp.company_id = cs.company_id
    WHERE jp.max_salary IS NOT NULL
    GROUP BY cs.speciality, jp.title
) t
WHERE rank <= 10
  AND speciality IN (
      -- Filtrer les spécialités ayant plus de 10 titres distincts
      SELECT speciality
      FROM LINKEDIN.GOLD.job_postings jp
      JOIN LINKEDIN.SILVER.company_specialities cs 
          ON jp.company_id = cs.company_id
      GROUP BY speciality
      HAVING COUNT(DISTINCT title) > 10
  )
ORDER BY speciality, rank;

## Répartition par taille d’entreprise

In [ ]:
SELECT 
    CASE 
        WHEN c.company_size BETWEEN 0 AND 2 THEN 'small_size'
        WHEN c.company_size > 2 AND c.company_size <= 4 THEN 'medium_size'
        WHEN c.company_size > 4 AND c.company_size <= 7 THEN 'big_size'
        ELSE 'unknown'
    END AS company_size_group,
    
    COUNT(*) AS nb_jobs,
    
    ROUND(
        100 * COUNT(*) / SUM(COUNT(*)) OVER (), 
        2
    ) AS pct

FROM LINKEDIN.GOLD.job_postings j
JOIN LINKEDIN.SILVER.companies c 
    ON j.company_id = c.company_id

GROUP BY company_size_group
ORDER BY company_size_group;

## Répartition par secteur

In [ ]:
SELECT 
    cs.speciality,
    COUNT(j.job_id) AS nb_jobs
FROM LINKEDIN.GOLD.job_postings j
JOIN LINKEDIN.SILVER.companies c 
    ON j.company_id = c.company_id
JOIN LINKEDIN.SILVER.Company_specialities cs 
    ON c.company_id = cs.company_id
GROUP BY cs.speciality
ORDER BY nb_jobs DESC;

## Répartition par type d’emploi

In [ ]:
SELECT 
    formatted_work_type,
    COUNT(*) AS nb_jobs
FROM LINKEDIN.GOLD.job_postings
GROUP BY 1
ORDER BY nb_jobs DESC;